In [ ]:
# IMPORT AUPassata Regular + parse config

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import py2bit
import numpy as np
import yaml

font_path = './AUPassata_Rg.ttf'
fm.fontManager.addfont(font_path)


def init_plotting():
    plt.rcParams['figure.figsize'] = (8, 3)
    plt.rcParams['font.size'] = 10
    plt.rcParams['font.family'] = 'AU Passata'
    plt.rcParams['axes.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['axes.titlesize'] = 1.5 * plt.rcParams['font.size']
    plt.rcParams['legend.fontsize'] = plt.rcParams['font.size']
    plt.rcParams['xtick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['ytick.labelsize'] = plt.rcParams['font.size']
    plt.rcParams['savefig.dpi'] = 200
    plt.rcParams['xtick.major.size'] = 3
    plt.rcParams['xtick.major.width'] = 1
    plt.rcParams['ytick.major.size'] = 3
    plt.rcParams['ytick.major.width'] = 1
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['axes.linewidth'] = 1

    ax = plt.gca()
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')


with open('config.yaml') as f:
    config = yaml.safe_load(f)

TRAINING_DHS_DIR = config['training_dhs_dir']

In [ ]:
LYMPHOID_INPUT = f'{TRAINING_DHS_DIR}Lymphoid.bed'

lymphoid_df = pd.read_csv(LYMPHOID_INPUT, sep='\t', names=['chr', 'start', 'end'])
lymphoid_df['mid'] = (lymphoid_df['start'] + lymphoid_df['end']) // 2
lymphoid_df['length'] = lymphoid_df['end'] - lymphoid_df['start']

init_plotting()
dhs_site_lengths = lymphoid_df['length']
mn, mx, median, avg, std = (
    min(dhs_site_lengths),
    max(dhs_site_lengths),
    dhs_site_lengths.median(),
    dhs_site_lengths.mean(),
    dhs_site_lengths.std(),
)
plt.hist(dhs_site_lengths, bins=100, edgecolor='black', alpha=0.8)

plt.plot([], [], ' ', label=f'Min site length = {mn:,}')
plt.plot([], [], ' ', label=f'Max site length = {mx:,}')
plt.plot([], [], ' ', label=f'Median site length = {median:.2f}')
plt.plot([], [], ' ', label=f'Average site length = {avg:.2f}')
plt.plot([], [], ' ', label=f'Std site length = {std:.2f}')

plt.legend(loc='upper right')
plt.xlabel('DHS site length [end - start]')
plt.ylabel('Frequency')
plt.title(f'DHS site length distribution\n({len(dhs_site_lengths):,} Lymphoid DHS sites)')

plt.tight_layout()
plt.show()

In [ ]:
hg38_2bit_path = config['hg_38_2bit_file']
W = 100
num_bases_per_window = 2 * W
LYMPHOID_INPUT = f'{TRAINING_DHS_DIR}Lymphoid.bed'
LYMPHOID_NEGATIVE_INPUT = f'{TRAINING_DHS_DIR}Lymphoid_negative.bed'

lymphoid_df = pd.read_csv(LYMPHOID_INPUT, sep='\t', names=['chr', 'start', 'end'])
lymphoid_df['mid'] = (lymphoid_df['start'] + lymphoid_df['end']) // 2
lymphoid_df['length'] = lymphoid_df['end'] - lymphoid_df['start']

lymphoid_negative_df = pd.read_csv(LYMPHOID_NEGATIVE_INPUT, sep='\t', names=['chr', 'start', 'end'])
lymphoid_negative_df['mid'] = (lymphoid_negative_df['start'] + lymphoid_negative_df['end']) // 2

hg38_genome = py2bit.open(hg38_2bit_path)

gc = np.empty(len(lymphoid_df))
for i, (chrom, mid) in enumerate(zip(lymphoid_df['chr'].to_numpy(), lymphoid_df['mid'].to_numpy())):
    lower, upper = int(mid) - W, int(mid) + W
    base_distr = hg38_genome.bases(chrom, lower, upper, False)
    gc[i] = (base_distr['G'] + base_distr['C']) / num_bases_per_window

lymphoid_df['gc_content'] = gc

gc = np.empty(len(lymphoid_negative_df))
for i, (chrom, mid) in enumerate(
    zip(lymphoid_negative_df['chr'].to_numpy(), lymphoid_negative_df['mid'].to_numpy())
):
    lower, upper = int(mid) - W, int(mid) + W
    base_distr = hg38_genome.bases(chrom, lower, upper, False)
    gc[i] = (base_distr['G'] + base_distr['C']) / num_bases_per_window

lymphoid_negative_df['gc_content'] = gc

fig, axes = plt.subplots(
    2,
    1,
    sharex=True,
    sharey=True,
    figsize=(8, 6),
)

dhs_site_gc_content = lymphoid_df['gc_content']
mn, mx, median, avg, std = (
    min(dhs_site_gc_content),
    max(dhs_site_gc_content),
    dhs_site_gc_content.median(),
    dhs_site_gc_content.mean(),
    dhs_site_gc_content.std(),
)
axes[0].hist(dhs_site_gc_content, bins=100, edgecolor='black', alpha=0.8)

axes[0].plot([], [], ' ', label=f'Min gc content = {mn:,}')
axes[0].plot([], [], ' ', label=f'Max gc content = {mx:,}')
axes[0].plot([], [], ' ', label=f'Median gc content = {median:.2f}')
axes[0].plot([], [], ' ', label=f'Average gc content = {avg:.2f}')
axes[0].plot([], [], ' ', label=f'Std gc content = {std:.2f}')

axes[0].legend(loc='upper right')
axes[0].set_ylabel('Frequency')
axes[0].set_title(
    f'Lymphoid DHS site GC content distribution\n({len(lymphoid_df):,} Lymphoid DHS sites)'
)


dhs_site_gc_content = lymphoid_negative_df['gc_content']
mn, mx, median, avg, std = (
    min(dhs_site_gc_content),
    max(dhs_site_gc_content),
    dhs_site_gc_content.median(),
    dhs_site_gc_content.mean(),
    dhs_site_gc_content.std(),
)
axes[1].hist(dhs_site_gc_content, bins=100, edgecolor='black', alpha=0.8)

axes[1].plot([], [], ' ', label=f'Min gc content = {mn:,}')
axes[1].plot([], [], ' ', label=f'Max gc content = {mx:,}')
axes[1].plot([], [], ' ', label=f'Median gc content = {median:.2f}')
axes[1].plot([], [], ' ', label=f'Average gc content = {avg:.2f}')
axes[1].plot([], [], ' ', label=f'Std gc content = {std:.2f}')

axes[1].legend(loc='upper right')
axes[1].set_xlabel('DHS site GC content')
axes[1].set_ylabel('Frequency')
axes[1].set_title(
    f'Negative Lymphoid DHS site GC content distribution (corrected GC bias)\n({len(lymphoid_negative_df):,} Negative lymphoid DHS sites)'
)

plt.tight_layout()
plt.show()